# Book Recommendation Model

## Imports and data loading

In [51]:
import warnings

warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import sklearn as skl
import pycountry
import plotly.express as px
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    KFold,
)
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    brier_score_loss,
    log_loss,
    f1_score,
    roc_auc_score,
    adjusted_rand_score,
    silhouette_score,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CUR_DIR = os.getcwd()
BOOKS_DATA_PATH = os.path.join(CUR_DIR, "books_data/books.csv")
RATINGS_DATA_PATH = os.path.join(CUR_DIR, "books_data/ratings.csv")
USERS_DATA_PATH = os.path.join(CUR_DIR, "books_data/users.csv")

df_books = pd.read_csv(BOOKS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\").drop(columns = ['Image-URL-S', 'Image-URL-M', 'Image-URL-L'])
df_ratings = pd.read_csv(RATINGS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"')
df_users = pd.read_csv(USERS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\", index_col=0)

print(f"Books data    loaded {len(df_books):,} rows, {df_books.shape[1]} columns.")
print(f"Ratings data  loaded {len(df_ratings):,} rows, {df_ratings.shape[1]} columns.")
print(f"Users data    loaded {len(df_users):,} rows, {df_users.shape[1]} columns.")

df_active_countries = df_users['Location'].str.split(',').str[-1].str.strip()
unique_active_countries = df_active_countries.unique()

Books data    loaded 271,379 rows, 5 columns.
Ratings data  loaded 1,149,780 rows, 3 columns.
Users data    loaded 278,858 rows, 2 columns.


In [52]:
"""
METHOD 1
"""
countries = []
for country in pycountry.countries:
    countries.append(country.name.lower())

mutual = []
for mutual_country in countries:
    if mutual_country in unique_active_countries:
        mutual.append(mutual_country)

#ADD MANUALLY countries not in mutual
mutual += "usa", "russia", "iran", "vietnam", "u.a.e", "turkey", "taiwan", "syria", "venezuela", "south korea", "czech republic"
 
total_users_method1 = 0
for country in mutual:
    total_users_method1 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning;            METHOD 1: {round(total_users_method1/len(df_users)*100, 2)}")

"""
METHOD 2
this is an additional validation step of the mutual list
"""
countries_counts = df_active_countries.value_counts()
counts_list = countries_counts[countries_counts>50].index.tolist()
counts_list.remove("")

total_users_method2 = 0
for country in counts_list:
    total_users_method2 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning; only using METHOD 2: {round(total_users_method2/len(df_users)*100, 2)}")
print(f"Countries, having more than 50 customers, excluded from the countries list are: {set(counts_list)- set(mutual)}")

df_users['Country'] = df_users['Location'].str.split(',').str[-1].str.strip().str.lower()
df_users['Country'] = df_users['Country'].apply(lambda x: x if x in mutual else "unknown")

Percentage of users kept after cleaning;            METHOD 1: 97.92
Percentage of users kept after cleaning; only using METHOD 2: 97.38
Countries, having more than 50 customers, excluded from the countries list are: {'españa', 'yugoslavia'}


## Data Cleaning

In [76]:
#merge all data into one table - df_main
df_main = pd.merge(df_ratings,df_users, how='left', on='User-ID')
df_main = pd.merge(df_main, df_books, how= 'left', on='ISBN')
df_main = df_main.drop(['Location','Country'], axis = 1)
#restricts data only to top publishers
top_publishers = df_main['Publisher'].value_counts().head(10).index.to_list()
df_main = df_main[df_main['Publisher'].isin(top_publishers)]

#creates book-read-count column
read_count = df_main['Book-Title'].value_counts()
read_count.name = 'book-read-count'

#creates author-read-count column
author_counts = df_main['Book-Author'].value_counts()
author_counts.name = 'author-read-count'
df_main = pd.merge(df_main, author_counts, how='left', on='Book-Author')
df_main = pd.merge(df_main, read_count, how='left', on='Book-Title')

# One row per ISBN for displaying book information at the end
df_book_list = (
    df_main[
        [
            'ISBN',
            'Book-Title',
            'Book-Author',
            'Year-Of-Publication',
            'Publisher',
            'book-read-count',
            'author-read-count'
        ]
    ]
    .drop_duplicates('ISBN')
    .copy()
)

print("df_main:", df_main.shape)
print("Unique users:", df_main['User-ID'].nunique())
print("Unique books:", df_main['ISBN'].nunique())


df_main: (243009, 10)
Unique users: 33866
Unique books: 32440


## Book ISBN search based on title for future actions

In [77]:
def search_books_by_title(search_text, n=10):
    """
    Search books by title so you can find the correct ISBN.
    """
    search_text = str(search_text).lower()

    results = df_book_list[
        df_book_list['Book-Title']
        .astype(str)
        .str.lower()
        .str.contains(search_text, na=False)
    ].copy()

    results = results.sort_values(
        by='book-read-count',
        ascending=False
    )

    return results.head(n)

In [78]:
search_books_by_title("flies", n=5) 

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
4190,0553277243,Time Flies,BILL COSBY,1988.0,Bantam,55,55
15423,0671744526,The Woman Lit By Fireflies,Jim Harrison,1991.0,Pocket,8,8
34380,0399506438,William Golding's Lord of the Flies (Casebook ...,William Golding,1983.0,Berkley Publishing Group,6,7
94890,0373028458,"When Love Flies By (Harlequin Romance, No 2845)",Jeanne Allan,1987.0,Harlequin,3,44
89951,0140283331,Lord of the Flies (Penguin Great Books of the ...,William Golding,1999.0,Penguin Books,1,7


## Co-reader vote recommender

In [ ]:
def recommender(seed_isbns, k=0):
    # Clean input ISBNs
    seed_isbns = [str(isbn).strip() for isbn in seed_isbns]
    seed_isbns = list(set(seed_isbns))

    # Read/interacted events from df_main
    read_events = df_main[["User-ID", "ISBN"]].drop_duplicates()

    # Keep only seed books that exist in df_main
    valid_seed_isbns = [
        isbn for isbn in seed_isbns
        if isbn in set(df_main["ISBN"])
    ]

    print("Input seed ISBNs:", seed_isbns)
    print("Valid seed ISBNs:", valid_seed_isbns)

    if len(valid_seed_isbns) == 0:
        raise ValueError("None of the seed ISBNs exist in df_main.")

    print("rating threshold:", k)

    liked_seed_ratings = df_main[(df_main["ISBN"].isin(valid_seed_isbns)) & (df_main["Book-Rating"] >= k)].copy()

    if liked_seed_ratings.empty:
          raise ValueError("No users rated the seed books above or equal to k.")

    user_seed_matches = (liked_seed_ratings.groupby("User-ID")["ISBN"].nunique())
    user_seed_matches.name = "liked_seed_count"
    user_seed_matches = user_seed_matches.reset_index()

    score_column = "liked_seed_count"
    final_score_name = "liked_seed_vote_score"
    print("Matched users:", len(user_seed_matches))

    # Get all other books read by matched users
    candidate_reads = pd.merge(read_events, user_seed_matches, how="inner", on="User-ID")

    # Do not recommend the input seed books
    candidate_reads = candidate_reads[~candidate_reads["ISBN"].isin(valid_seed_isbns)].copy()
    if candidate_reads.empty:
        return pd.DataFrame()

    # Score candidate books
    scores = (
        candidate_reads
        .groupby("ISBN")
        .agg(
            vote_score=(score_column, "sum"),
            matched_reader_count=("User-ID", "nunique")
        ).reset_index())
    scores = scores.rename(columns={"vote_score": final_score_name})

    # Global reader count
    global_reader_counts = (read_events.groupby("ISBN")["User-ID"].nunique())

    global_reader_counts.name = "global_reader_count"
    global_reader_counts = global_reader_counts.reset_index()

    scores = pd.merge(scores, global_reader_counts, how="left", on="ISBN")
    scores = scores.sort_values(by=[final_score_name, "matched_reader_count"], ascending=False)

    recommendations = pd.merge(scores, df_book_list, how="left", on="ISBN")

    return recommendations

## Test

In [80]:
seed_isbns = [
    "0451139712",   # The Stand
    "0451157443",   # Carrie
    "0743424425	",  # The Shining
    "0451184963",   # Insomnia
    "044021145X",   # The Firm
    "0345337662",   # Interview with the Vampire
    "0440224675",   # Hannibal
    "0399501487",   # Lord of the Flies
    "0671027360",   # Angels and Demons

]

In [81]:
recommender(seed_isbns, k=0)

Input seed ISBNs: ['044021145X', '0440224675', '0743424425', '0671027360', '0399501487', '0451139712', '0345337662', '0451157443', '0451184963']
Valid seed ISBNs: ['0743424425', '0345337662', '0451157443', '0451184963']
Mode: co-reader votes based on read/interacted seed books
Matched users: 720


,ISBN,coreader_vote_score,matched_reader_count,global_reader_count,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,0345313860,194,172,301,"The Vampire Lestat (Vampire Chronicles, Book II)",ANNE RICE,1986.0,Ballantine Books,301,656
1,0345351525,162,141,273,The Queen of the Damned (Vampire Chronicles (P...,Anne Rice,1993.0,Ballantine Books,273,2219
2,0345370775,136,111,466,Jurassic Park,Michael Crichton,1999.0,Ballantine Books,468,1914
3,034538475X,113,99,193,The Tale of the Body Thief (Vampire Chronicles...,Anne Rice,1993.0,Ballantine Books,193,2219
4,0446672211,102,86,585,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,1998.0,Warner Books,585,819
...,...,...,...,...,...,...,...,...,...,...
18747,2266096400,1,1,1,Le CrÃ?Â©puscule des elfes,Jean-Louis Fetjaine,2002.0,Pocket,1,1
18748,2266102931,1,1,1,Fritna,GisÃ?Â¨le Halimi,2001.0,Pocket,1,1
18749,2266105701,1,1,1,Les mauvaises pensÃ?Â©es,Laurent Seksik,2001.0,Pocket,1,1
18750,2266107534,1,1,3,La citÃ?Â© de la joie,Dominique Lapierre,2000.0,Pocket,3,26


In [82]:
recommender(seed_isbns,k=8)

Input seed ISBNs: ['044021145X', '0440224675', '0743424425', '0671027360', '0399501487', '0451139712', '0345337662', '0451157443', '0451184963']
Valid seed ISBNs: ['0743424425', '0345337662', '0451157443', '0451184963']
Mode: liked-seed votes based on seed ratings >= k
k threshold: 8
Matched users: 218


,ISBN,liked_seed_vote_score,matched_reader_count,global_reader_count,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,0345313860,60,59,301,"The Vampire Lestat (Vampire Chronicles, Book II)",ANNE RICE,1986.0,Ballantine Books,301,656
1,0345351525,48,47,273,The Queen of the Damned (Vampire Chronicles (P...,Anne Rice,1993.0,Ballantine Books,273,2219
2,034538475X,40,39,193,The Tale of the Body Thief (Vampire Chronicles...,Anne Rice,1993.0,Ballantine Books,193,2219
3,0345370775,29,28,466,Jurassic Park,Michael Crichton,1999.0,Ballantine Books,468,1914
4,0451156609,29,25,170,The Tommyknockers,Stephen King,1994.0,Signet Book,170,5667
...,...,...,...,...,...,...,...,...,...,...
6675,2266118536,1,1,3,Myrtille Ã?Â la plage,Olivier Mau,2003.0,Pocket,3,4
6676,2266120166,1,1,11,La Rage au coeur,Ingrid Betancourt,2002.0,Pocket,11,11
6677,2266130250,1,1,2,"Rupture dans le rÃ?Â©el, tome 1-1 : GÃ?Â©nÃ?Â©se",Peter F. Hamilton,2003.0,Pocket,2,24
6678,2266131516,1,1,4,La MÃ?Â©thode simple pour en finir avec la cig...,Allen Carr,2003.0,Pocket,4,4


In [83]:
def test_random_user_recommendations(
    min_books_read=15,
    n_seed_books=15,
    random_state=42,
    use_liked_seed_model=False,
    k=8
):
    # Create read/interacted events from df_main
    read_events = df_main[["User-ID", "ISBN"]].drop_duplicates()

    # Count how many books each user has read/interacted with
    user_book_counts = read_events.groupby("User-ID")["ISBN"].nunique()

    # Keep only users with at least min_books_read books
    eligible_users = user_book_counts[user_book_counts >= min_books_read].index

    print("Eligible users:", len(eligible_users))

    if len(eligible_users) == 0:
        raise ValueError("No users found with enough read books.")

    # Pick one random user
    random_user_id = pd.Series(eligible_users).sample(
        n=1,
        random_state=random_state
    ).iloc[0]

    print("Random user:", random_user_id)

    # Get all books read by this user
    random_user_books = read_events[
        read_events["User-ID"] == random_user_id
    ].copy()

    # Add book information to all read books
    random_user_books_info = random_user_books.merge(
        df_book_list,
        on="ISBN",
        how="left"
    )

    # Pick n_seed_books random books from this user as input
    seed_books = random_user_books.sample(
        n=n_seed_books,
        random_state=random_state
    ).copy()

    seed_isbns = seed_books["ISBN"].tolist()

    # Add book information to seed books
    seed_books_info = seed_books.merge(
        df_book_list,
        on="ISBN",
        how="left"
    )

    print("\nSeed ISBNs:")
    print(seed_isbns)

    print("\nBooks used as input:")
    display(seed_books_info)

    # Run recommendation model
    if use_liked_seed_model:
        recommendations = recommender(seed_isbns, k=k)
    else:
        recommendations = recommender(seed_isbns, k=0)

    print("\nRecommendations:")
    display(recommendations)


    return random_user_id, seed_books_info, recommendations, random_user_books_info

In [84]:
random_user_id, seed_books_info, recommendations, random_user_books_info = test_random_user_recommendations(
    min_books_read=30,
    n_seed_books=20,
    random_state=42,
    use_liked_seed_model=False
)

Eligible users: 1343
Random user: 174216

Seed ISBNs:
['0671727737', '0671458914', '0060009241', '0451197860', '0345372050', '0446605336', '0380718340', '0345438329', '0425184129', '067166641X', '0553582801', '0345423291', '0451180062', '0345450914', '0446527785', '0553265962', '0345367421', '0380726815', '0553213067', '0345413873']

Books used as input:


,User-ID,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,174216,0671727737,The Eagle Has Landed,Jack Higgins,1990.0,Pocket,18,575
1,174216,0671458914,FDR CENTENRY REMBR,Kelli M. Gary,1982.0,Pocket,1,157
2,174216,0060009241,See Jane Score (Avon Romance),Rachel Gibson,2003.0,Avon,49,203
3,174216,0451197860,Mothers and Daughters,Jill Morgan,1999.0,Signet Book,13,13
4,174216,0345372050,You Just Don't Understand,DEBORAH TANNEN,1991.0,Ballantine Books,37,38
5,174216,0446605336,My Sergei : A Love Story,E. M. Swift,1997.0,Warner Books,33,33
6,174216,0380718340,Cruel &amp; Unusual (Kay Scarpetta Mysteries (...,Patricia D. Cornwell,1994.0,Avon,199,518
7,174216,0345438329,Big Stone Gap: A Novel (Ballantine Reader's Ci...,Adriana Trigiani,2001.0,Ballantine Books,182,285
8,174216,0425184129,Big Trouble,Dave Barry,2002.0,Berkley Publishing Group,172,400
9,174216,067166641X,SURRENDER THE PINK : SURRENDER THE PINK,Angela Fisher,1991.0,Pocket,30,65


Input seed ISBNs: ['0060009241', '0671727737', '0380718340', '0425184129', '067166641X', '0446527785', '0345450914', '0671458914', '0345423291', '0345413873', '0345372050', '0380726815', '0553213067', '0451180062', '0345438329', '0345367421', '0451197860', '0446605336', '0553265962', '0553582801']
Valid seed ISBNs: ['0060009241', '0671727737', '0380718340', '0425184129', '067166641X', '0446527785', '0345450914', '0671458914', '0345423291', '0345413873', '0345372050', '0380726815', '0553213067', '0451180062', '0345438329', '0345367421', '0451197860', '0446605336', '0553265962', '0553582801']
Mode: co-reader votes based on read/interacted seed books
Matched users: 883

Recommendations:


,ISBN,coreader_vote_score,matched_reader_count,global_reader_count,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,0446672211,210,128,585,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,1998.0,Warner Books,585,819
1,0142001740,190,112,615,The Secret Life of Bees,Sue Monk Kidd,2003.0,Penguin Books,615,615
2,0446605239,186,114,465,The Notebook,Nicholas Sparks,1998.0,Warner Books,647,2191
3,044651652X,179,84,368,The Bridges of Madison County,Robert James Waller,1992.0,Warner Books,426,910
4,0345370775,171,99,466,Jurassic Park,Michael Crichton,1999.0,Ballantine Books,468,1914
...,...,...,...,...,...,...,...,...,...,...
20206,226603071X,1,1,1,"Les Liaisons Dangereuses (Fiction, Poetry and ...",Laclos,0.0,Pocket,1,1
20207,2266078720,1,1,2,Meurtres Ã?Â Gramercy Park,Carol O'Connell,1999.0,Pocket,2,25
20208,2266096400,1,1,1,Le CrÃ?Â©puscule des elfes,Jean-Louis Fetjaine,2002.0,Pocket,1,1
20209,2266110527,1,1,1,Le Sphinx,Graham Masterton,2001.0,Pocket,1,18


In [85]:
random_user_id, seed_books_info, recommendations, random_user_books_info = test_random_user_recommendations(
    min_books_read=30,
    n_seed_books=20,
    random_state=42,
    use_liked_seed_model=True,
    k=8
)

Eligible users: 1343
Random user: 174216

Seed ISBNs:
['0671727737', '0671458914', '0060009241', '0451197860', '0345372050', '0446605336', '0380718340', '0345438329', '0425184129', '067166641X', '0553582801', '0345423291', '0451180062', '0345450914', '0446527785', '0553265962', '0345367421', '0380726815', '0553213067', '0345413873']

Books used as input:


,User-ID,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,174216,0671727737,The Eagle Has Landed,Jack Higgins,1990.0,Pocket,18,575
1,174216,0671458914,FDR CENTENRY REMBR,Kelli M. Gary,1982.0,Pocket,1,157
2,174216,0060009241,See Jane Score (Avon Romance),Rachel Gibson,2003.0,Avon,49,203
3,174216,0451197860,Mothers and Daughters,Jill Morgan,1999.0,Signet Book,13,13
4,174216,0345372050,You Just Don't Understand,DEBORAH TANNEN,1991.0,Ballantine Books,37,38
5,174216,0446605336,My Sergei : A Love Story,E. M. Swift,1997.0,Warner Books,33,33
6,174216,0380718340,Cruel &amp; Unusual (Kay Scarpetta Mysteries (...,Patricia D. Cornwell,1994.0,Avon,199,518
7,174216,0345438329,Big Stone Gap: A Novel (Ballantine Reader's Ci...,Adriana Trigiani,2001.0,Ballantine Books,182,285
8,174216,0425184129,Big Trouble,Dave Barry,2002.0,Berkley Publishing Group,172,400
9,174216,067166641X,SURRENDER THE PINK : SURRENDER THE PINK,Angela Fisher,1991.0,Pocket,30,65


Input seed ISBNs: ['0060009241', '0671727737', '0380718340', '0425184129', '067166641X', '0446527785', '0345450914', '0671458914', '0345423291', '0345413873', '0345372050', '0380726815', '0553213067', '0451180062', '0345438329', '0345367421', '0451197860', '0446605336', '0553265962', '0553582801']
Valid seed ISBNs: ['0060009241', '0671727737', '0380718340', '0425184129', '067166641X', '0446527785', '0345450914', '0671458914', '0345423291', '0345413873', '0345372050', '0380726815', '0553213067', '0451180062', '0345438329', '0345367421', '0451197860', '0446605336', '0553265962', '0553582801']
Mode: liked-seed votes based on seed ratings >= k
k threshold: 8
Matched users: 233

Recommendations:


,ISBN,liked_seed_vote_score,matched_reader_count,global_reader_count,Book-Title,Book-Author,Year-Of-Publication,Publisher,book-read-count,author-read-count
0,0446672211,46,40,585,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,1998.0,Warner Books,585,819
1,0142001740,37,30,615,The Secret Life of Bees,Sue Monk Kidd,2003.0,Penguin Books,615,615
2,0425147622,35,31,222,The Body Farm,Patricia Daniels Cornwell,1995.0,Berkley Publishing Group,225,1602
3,0345443284,33,26,365,While I Was Gone,Sue Miller,1999.0,Ballantine Books,430,416
4,0380718332,32,29,184,All That Remains (Kay Scarpetta Mysteries (Pap...,Patricia D. Cornwell,1993.0,Avon,184,518
...,...,...,...,...,...,...,...,...,...,...
9904,1572972971,1,1,4,"Jedi Bounty (Star Wars: Young Jedi Knights, Bo...",Kevin J. Anderson,1997.0,Berkley Publishing Group,4,242
9905,1572973234,1,1,8,"Love, Lucy",Lucille Ball,1997.0,Berkley Publishing Group,9,9
9906,1572973315,1,1,4,The Emperor's Plague (Star Wars: Young Jedi Kn...,Kevin J. Anderson,1998.0,Berkley Publishing Group,4,242
9907,2266105329,1,1,1,Accident,Danielle Steel,2001.0,Pocket,1,85
